# PHA - Full Pipeline Demo

This notebook demonstrates the complete **Personal Health Insights Agent Team (PHA)** pipeline.

The orchestrator coordinates three specialized agents:
1. **Data Science Agent**: Analyzes health data, trends, and patterns
2. **Domain Expert Agent**: Provides medical interpretation and context
3. **Health Coach Agent**: Delivers conversational guidance and recommendations

## How It Works

1. User asks a health question
2. Orchestrator determines which agents should handle it
3. Supporting agents provide insights (potentially in parallel)
4. Main agent synthesizes a response
5. User receives a comprehensive, personalized answer

## Setup

In [ ]:
import os
import sys

# Add the project root to the path
sys.path.insert(0, '..')

# --- Configuration ---
# Set your API key and provider here
API_KEY = "your-api-key-here"
# Provider options: "gemini", "openai", "anthropic"
PROVIDER = "gemini"

# Optional: Tavily API key for Domain Expert Agent (search)
TAVILY_API_KEY = "your-tavily-api-key-here"


In [ ]:
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    MultiAgentOrchestrator,
    is_react_available,
    create_orchestrator,
)

print("All agents imported successfully!")
print(f"ReAct available for Domain Expert: {is_react_available()}")

## Load Sample Health Data

In [ ]:
import pandas as pd

# Load sample data
summary_df = pd.read_csv('../data/sample/summary.csv')
activities_df = pd.read_csv('../data/sample/activities.csv')
profile_df = pd.read_csv('../data/sample/profile.csv')
population_df = pd.read_csv('../data/sample/population_percentiles.csv')

print(f"Loaded data:")
print(f"  - Summary: {len(summary_df)} days")
print(f"  - Activities: {len(activities_df)} records")
print(f"  - Profile: {profile_df.to_dict('records')[0]}")

## Initialize Individual Agents

In [ ]:
# Initialize Data Science Agent
ds_agent = DataScienceAgent()
ds_agent.configure(api_key=API_KEY, provider=PROVIDER) # Implicitly uses global config
ds_agent.load_dataframes({
    'summary': summary_df,
    'activities': activities_df,
    'profile': profile_df,
    'population': population_df,
})
print("✓ Data Science Agent initialized")

In [ ]:
# Initialize Domain Expert Agent (if ReAct available)
de_agent = None
if is_react_available():
    import glob
    de_agent = DomainExpertAgent(
        search_backend='tavily',  # or 'duckduckgo' for free search
        tavily_api_key=TAVILY_API_KEY,
    )
    exemplar_files = glob.glob('../few_shots/*.ipynb')
    de_agent.get_agent(
        api_key=API_KEY,
        provider=PROVIDER,
        exemplar_files=exemplar_files,
    )
    print(f"✓ Domain Expert Agent initialized with {len(exemplar_files)} exemplars")
else:
    print("⚠ Domain Expert Agent unavailable (install onetwo for ReAct support)")


In [ ]:
# Initialize Health Coach Agent
hc_agent = HealthCoachAgent(simple_mode=True)
hc_agent.configure(api_key=API_KEY, provider=PROVIDER)
print("✓ Health Coach Agent initialized")

## Initialize the Orchestrator

In [ ]:
# Create the orchestrator and connect all agents
orchestrator = MultiAgentOrchestrator(debug_verbose=True)
orchestrator.configure(api_key=API_KEY, provider=PROVIDER)
orchestrator.set_agents(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=hc_agent,
)

print("\n✓ Orchestrator initialized!")
print(f"  Agents: {orchestrator.agent_name_list}")

## Test the Full Pipeline

In [ ]:
# Ask a comprehensive health question
response = orchestrator.respond(
    "Based on my health data, how is my sleep quality and what can I do to improve it?"
)

print("\n" + "="*60)
print("RESPONSE:")
print("="*60)
print(response)

In [ ]:
# Check the team structure that was used
print("Team Structure Used:")
print(orchestrator.get_team_structure())

In [ ]:
# Check individual agent insights
print("\nAgent Insights:")
insights = orchestrator.get_agent_insights()
for agent, insight in insights.items():
    if insight:
        print(f"\n--- {agent.upper()} ---")
        print(insight[:500] + "..." if len(insight) > 500 else insight)

## Multi-Turn Conversation

In [ ]:
# Continue the conversation
response = orchestrator.respond(
    "How does my activity level compare to others my age?"
)

print("\n" + "="*60)
print("RESPONSE:")
print("="*60)
print(response)

In [ ]:
# Ask about trends
response = orchestrator.respond(
    "Have my resting heart rate or steps been trending up or down lately?"
)

print("\n" + "="*60)
print("RESPONSE:")
print("="*60)
print(response)

## View Conversation History

In [ ]:
print("Conversation History:")
print(orchestrator.get_conversation_history())

## Quick Factory Method

For convenience, you can use the factory function to create a fully configured orchestrator:

In [ ]:
# Quick setup using factory function
quick_orchestrator = create_orchestrator(
    data_science_agent=ds_agent,
    domain_expert_agent=de_agent,
    health_coach_agent=hc_agent,
    debug_verbose=False,  # Quiet mode
)

response = quick_orchestrator.respond("What's my average daily step count?")
print(response)

## Summary

The PHA orchestrator:

1. **Automatically routes queries** to the most appropriate agents
2. **Coordinates parallel calls** to supporting agents for efficiency
3. **Synthesizes insights** through the main agent
4. **Maintains conversation context** across turns
5. **Provides comprehensive answers** by combining data analysis, medical expertise, and coaching

### Architecture

```
User Question
     ↓
Orchestrator
     ↓
┌────────────────┬─────────────────┬──────────────────┐
│ Data Science   │ Domain Expert   │ Health Coach     │
│ Agent          │ Agent           │ Agent            │
│ (analyze data) │ (interpret)     │ (guide/coach)    │
└────────────────┴─────────────────┴──────────────────┘
     ↓                   ↓                  ↓
     └──────────────────┴─────────────────┘
                         ↓
                 Synthesized Response
                         ↓
                       User
```